# 🤖 Proyecto Final — Modelo de Aprendizaje Supervisado
### Predicción de Propina — Regresión
**Asignatura:** Introducción a la Ciencia de Datos — ITM  
**Variable objetivo:** `Propina` (pesos COP)  
**Tipo de problema:** Regresión


## 1. Importar Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)


## 2. Carga del Dataset Limpio

In [ ]:
df = pd.read_csv("cafeteria_limpio.csv")
print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Variable objetivo — Propina (COP):")
print(f"  Media   : {df['Propina'].mean():,.0f}")
print(f"  Mediana : {df['Propina'].median():,.0f}")
print(f"  Rango   : [{df['Propina'].min():,}, {df['Propina'].max():,}]")
df.head(3)


## 3. Ingeniería de Características

In [ ]:
df_model = df.copy()

# Convertir Fecha a características temporales
df_model['Fecha'] = pd.to_datetime(df_model['Fecha'])
df_model['Año'] = df_model['Fecha'].dt.year
df_model['Mes'] = df_model['Fecha'].dt.month
df_model['DiaSemana'] = df_model['Fecha'].dt.dayofweek

# Codificar variables categóricas con LabelEncoder
cat_cols = ['Vendedor', 'Tipo', 'Producto', 'Categoria', 'Tipode', 'Tipo de pago',
            'Departamento', 'Municipio', 'Zona']

le = LabelEncoder()
for col in cat_cols:
    df_model[col + '_enc'] = le.fit_transform(df_model[col].astype(str))

# Seleccionar features y target
features = ['Precio', 'Cantidad', 'Mesa', 'Hora de Cobro',
            'Año', 'Mes', 'DiaSemana',
            'Vendedor_enc', 'Tipo_enc', 'Categoria_enc',
            'Tipode_enc', 'Tipo de pago_enc', 'Zona_enc']

X = df_model[features]
y = df_model['Propina']

print(f"Features seleccionadas: {len(features)}")
print(f"X shape: {X.shape} | y shape: {y.shape}")


## 4. División Train / Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Escalar
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]:,} muestras ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test : {X_test.shape[0]:,}  muestras ({X_test.shape[0]/len(X)*100:.0f}%)")


## 5. Modelo Base — Random Forest Regressor

In [ ]:
rf_base = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_base.fit(X_train, y_train)
y_pred_base = rf_base.predict(X_test)

mse_base  = mean_squared_error(y_test, y_pred_base)
rmse_base = np.sqrt(mse_base)
mae_base  = mean_absolute_error(y_test, y_pred_base)
r2_base   = r2_score(y_test, y_pred_base)

print("=" * 45)
print("  MÉTRICAS MODELO BASE (Random Forest)")
print("=" * 45)
print(f"  MSE  : {mse_base:>12,.2f}")
print(f"  RMSE : {rmse_base:>12,.2f}")
print(f"  MAE  : {mae_base:>12,.2f}")
print(f"  R²   : {r2_base:>12.4f}")
print("=" * 45)


In [ ]:
# Importancia de variables
feat_imp = pd.Series(rf_base.feature_importances_, index=features).sort_values(ascending=False)

plt.figure(figsize=(10, 5))
feat_imp.head(10).plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Top 10 Variables más Importantes — Random Forest Base')
plt.xlabel('Importancia')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('importancia_variables.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Optimización de Hiperparámetros — GridSearchCV

In [ ]:
# Usamos muestra para agilizar el GridSearch
sample_idx = np.random.choice(len(X_train), size=5000, replace=False)
X_sample = X_train.iloc[sample_idx]
y_sample = y_train.iloc[sample_idx]

param_grid = {
    'n_estimators'     : [100, 200],
    'max_depth'        : [10, 20, None],
    'min_samples_split': [2, 5],
    'min_samples_leaf' : [1, 2]
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42, n_jobs=-1),
    param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_sample, y_sample)
print(f"\n✅ Mejores hiperparámetros:")
print(grid_search.best_params_)
print(f"   Mejor R² en CV: {grid_search.best_score_:.4f}")


## 7. Modelo Optimizado — Evaluación Final

In [ ]:
best_params = grid_search.best_params_
rf_opt = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
rf_opt.fit(X_train, y_train)
y_pred_opt = rf_opt.predict(X_test)

mse_opt  = mean_squared_error(y_test, y_pred_opt)
rmse_opt = np.sqrt(mse_opt)
mae_opt  = mean_absolute_error(y_test, y_pred_opt)
r2_opt   = r2_score(y_test, y_pred_opt)

print("=" * 52)
print("  COMPARACIÓN: BASE vs OPTIMIZADO")
print("=" * 52)
print(f"{'Métrica':<10} {'Base':>15} {'Optimizado':>15}")
print("-" * 52)
print(f"{'RMSE':<10} {rmse_base:>15,.2f} {rmse_opt:>15,.2f}")
print(f"{'MAE':<10} {mae_base:>15,.2f} {mae_opt:>15,.2f}")
print(f"{'R²':<10} {r2_base:>15.4f} {r2_opt:>15.4f}")
print("=" * 52)
mejora = (r2_opt - r2_base) / abs(r2_base) * 100
print(f"\n  Mejora en R²: {mejora:+.2f}%")


In [ ]:
# Gráfico Predicho vs Real
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title in zip(
    axes,
    [y_pred_base, y_pred_opt],
    ['Modelo Base', 'Modelo Optimizado']
):
    ax.scatter(y_test[:500], y_pred[:500], alpha=0.4, s=15, color='steelblue')
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', linewidth=1.5, label='Ideal')
    ax.set_xlabel('Propina Real (COP)')
    ax.set_ylabel('Propina Predicha (COP)')
    ax.set_title(f'{title} — Predicho vs Real')
    ax.legend()

plt.suptitle('Evaluación Visual del Modelo', fontsize=13)
plt.tight_layout()
plt.savefig('predicho_vs_real.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Distribución de residuos
residuos = y_test - y_pred_opt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(residuos, bins=40, color='mediumseagreen', edgecolor='white')
axes[0].axvline(0, color='red', linestyle='--')
axes[0].set_title('Distribución de Residuos (Modelo Optimizado)')
axes[0].set_xlabel('Error (Real − Predicho)')
axes[0].set_ylabel('Frecuencia')

axes[1].scatter(y_pred_opt[:500], residuos[:500], alpha=0.4, s=15, color='mediumseagreen')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Propina Predicha (COP)')
axes[1].set_ylabel('Residuo')
axes[1].set_title('Residuos vs Predichos')

plt.tight_layout()
plt.savefig('residuos.png', dpi=150, bbox_inches='tight')
plt.show()

import scipy.stats as stats
_, p_value = stats.normaltest(residuos)
print(f"Prueba de normalidad de residuos — p-value: {p_value:.4f}")
print("→ Residuos con distribución normal ✅" if p_value > 0.05 else "→ Residuos no normales (revisar)")


## 8. Conclusiones del Modelo

In [ ]:
print("=" * 60)
print("  RESUMEN FINAL DEL MODELO")
print("=" * 60)
print(f"  Algoritmo       : Random Forest Regressor")
print(f"  Variable objetivo: Propina (COP)")
print(f"  Muestras train  : {len(X_train):,}")
print(f"  Muestras test   : {len(X_test):,}")
print(f"  Mejores params  : {best_params}")
print(f"  R² final        : {r2_opt:.4f}")
print(f"  RMSE final      : {rmse_opt:,.2f} COP")
print(f"  MAE final       : {mae_opt:,.2f} COP")
print("=" * 60)
print(f"\n  Interpretación:")
print(f"  El modelo explica el {r2_opt*100:.1f}% de la varianza de la propina.")
print(f"  El error promedio de predicción es de {mae_opt:,.0f} COP.")
